In [ ]:
%pip install -q matplotlib numpy pandas

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt

# -----------------------------
# LaTeX layout (measured)
# -----------------------------
# COLUMNWIDTH = 252.0 pt
# TEXTWIDTH   = 516.0 pt

PT_TO_INCH = 1 / 72.27
COLUMNWIDTH_IN = 252.0 * PT_TO_INCH  # ≈ 3.487 in
TEXTWIDTH_IN = 516.0 * PT_TO_INCH  # ≈ 7.141 in
HEIGHT_RATIO = 0.618
print(
    COLUMNWIDTH_IN,
    TEXTWIDTH_IN,
    COLUMNWIDTH_IN * HEIGHT_RATIO,
    TEXTWIDTH_IN * HEIGHT_RATIO,
)

# -----------------------------
# Global font sizes (10 pt paper)
# -----------------------------
FONT_SIZE = 9
SMALL_FONT_SIZE = 8

plt.rcParams.update(
    {
        'font.size': FONT_SIZE,
        'axes.labelsize': FONT_SIZE,
        'axes.titlesize': FONT_SIZE,
        'xtick.labelsize': SMALL_FONT_SIZE,
        'ytick.labelsize': SMALL_FONT_SIZE,
        'legend.fontsize': SMALL_FONT_SIZE,
        'pdf.fonttype': 42,  # editable text in PDF
        'ps.fonttype': 42,
    },
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import ScalarFormatter

# GPFS fileset scaling

In [ ]:
data = [
    # filesets, clients, partitions, mode, changelogs_per_s
    (1, 1, 1, 'changelog', 57935.19637),
    (1, 1, 1, 'icicle', 55701.38792),
    (1, 1, 1, 'icicle+red', 62937.62398),
    (1, 1, 2, 'changelog', 60377.28946),
    (1, 1, 2, 'icicle', 55080.51250),
    (1, 1, 2, 'icicle+red', 63570.00239),
    (2, 2, 1, 'changelog', 114652.2583),
    (2, 2, 1, 'icicle', 111438.0403),
    (2, 2, 1, 'icicle+red', 124215.8517),
    (2, 2, 2, 'changelog', 120511.9734),
    (2, 2, 2, 'icicle', 109013.0703),
    (2, 2, 2, 'icicle+red', 126201.8066),
    (3, 3, 1, 'changelog', 171967.1498),
    (3, 3, 1, 'icicle', 166498.6156),
    (3, 3, 1, 'icicle+red', 187430.1526),
    (3, 3, 2, 'changelog', 179494.4927),
    (3, 3, 2, 'icicle', 161414.4316),
    (3, 3, 2, 'icicle+red', 190165.3168),
    (4, 4, 1, 'changelog', 231703.5861),
    (4, 4, 1, 'icicle', 223656.3930),
    (4, 4, 1, 'icicle+red', 250668.6043),
    (4, 4, 2, 'changelog', 234149.4417),
    (4, 4, 2, 'icicle', 215955.5783),
    (4, 4, 2, 'icicle+red', 238941.0679),
    (5, 5, 1, 'changelog', 290562.6382),
    (5, 5, 1, 'icicle', 279499.6538),
    (5, 5, 1, 'icicle+red', 315683.3579),
    (5, 5, 2, 'changelog', 292273.5299),
    (5, 5, 2, 'icicle', 273243.6619),
    (5, 5, 2, 'icicle+red', 307036.4605),
]

df = pd.DataFrame(
    data,
    columns=[
        'num_filesets',
        'num_clients',
        'topic_partitions',
        'mode',
        'changelogs_per_s',
    ],
)

df['server'] = 'c5a.xlarge'
df['client_type'] = 'c5a.large'

# df

In [ ]:
lines = [
    (1, 'icicle'),
    (1, 'icicle+red'),
    (2, 'icicle'),
    (2, 'icicle+red'),
]

# Exact single-column figure size
fig, ax = plt.subplots(figsize=(COLUMNWIDTH_IN, COLUMNWIDTH_IN * HEIGHT_RATIO))

line_handles = {}

for partitions, mode in lines:
    subset = df[
        (df['topic_partitions'] == partitions) & (df['mode'] == mode)
    ].sort_values('num_clients')

    (line,) = ax.plot(
        subset['num_clients'],
        subset['changelogs_per_s'],
        marker='o',
        linewidth=1.8,
        label=f'{mode}, {partitions}p',
    )

    line_handles[(mode, partitions)] = line

# ax.set_title("Icicle Throughput")
# ---- Axes ----
ax.set_xlabel('Number of Clients')
ax.set_xticks([1, 2, 3, 4, 5])

yticks = [0, 100000, 200000, 300000, 400000]
ax.set_yticks(yticks)
ax.set_yticklabels([f'{y // 1000}K' for y in yticks])
ax.set_ylim(0, 400000)
ax.set_ylabel('Throughput\n(changelogs/s)')

# ---- Grid ----
# ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
# ax.set_axisbelow(True)

ax.yaxis.set_minor_locator(MultipleLocator(50000))
ax.grid(axis='y', which='major', linestyle='--', linewidth=0.4, alpha=0.5)
ax.grid(axis='y', which='minor', linestyle=':', linewidth=0.3, alpha=0.4)
ax.set_axisbelow(True)

# ---- Legends ----
legend_icicle = ax.legend(
    [
        line_handles[('icicle', 1)],
        line_handles[('icicle', 2)],
    ],
    [
        'Icicle (1p)',
        'Icicle (2p)',
    ],
    loc='lower right',
    frameon=False,
)

legend_icicle_red = ax.legend(
    [
        line_handles[('icicle+red', 1)],
        line_handles[('icicle+red', 2)],
    ],
    [
        'Icicle+Red. (1p)',
        'Icicle+Red. (2p)',
    ],
    loc='upper left',
    frameon=False,
)

ax.add_artist(legend_icicle)

# ---- Save ----
fig.tight_layout()
fig.savefig(
    'figures/gpfs_scaling_clients.pdf',
    bbox_inches='tight',
    dpi=600,
)
plt.show()

# GPFS scaling server size and client size

In [ ]:
import pandas as pd

data = [
    # 1p
    ('1p', 'changelog', 'c5a.xlarge', 73819),
    ('1p', 'changelog', 'c5a.2xlarge', 74674),
    ('1p', 'changelog', 'c5a.4xlarge', 74736),
    ('1p', 'changelog', 'c5a.8xlarge', 75407),
    ('1p', 'changelog', 'c5a.16xlarge', 76692),
    ('1p', 'icicle', 'c5a.xlarge', 73581),
    ('1p', 'icicle', 'c5a.2xlarge', 74566),
    ('1p', 'icicle', 'c5a.4xlarge', 74242),
    ('1p', 'icicle', 'c5a.8xlarge', 76635),
    ('1p', 'icicle', 'c5a.16xlarge', 75997),
    ('1p', 'icicle+red', 'c5a.xlarge', 76372),
    ('1p', 'icicle+red', 'c5a.2xlarge', 76912),
    ('1p', 'icicle+red', 'c5a.4xlarge', 77028),
    ('1p', 'icicle+red', 'c5a.8xlarge', 76711),
    ('1p', 'icicle+red', 'c5a.16xlarge', 76658),
    # 2p
    ('2p', 'changelog', 'c5a.xlarge', 110909),
    ('2p', 'changelog', 'c5a.2xlarge', 142177),
    ('2p', 'changelog', 'c5a.4xlarge', 139932),
    ('2p', 'changelog', 'c5a.8xlarge', 144921),
    ('2p', 'changelog', 'c5a.16xlarge', 140000),
    ('2p', 'icicle', 'c5a.xlarge', 92644),
    ('2p', 'icicle', 'c5a.2xlarge', 105685),
    ('2p', 'icicle', 'c5a.4xlarge', 106810),
    ('2p', 'icicle', 'c5a.8xlarge', 110113),
    ('2p', 'icicle', 'c5a.16xlarge', 108710),
    ('2p', 'icicle+red', 'c5a.xlarge', 113613),
    ('2p', 'icicle+red', 'c5a.2xlarge', 133830),
    ('2p', 'icicle+red', 'c5a.4xlarge', 138653),
    ('2p', 'icicle+red', 'c5a.8xlarge', 141089),
    ('2p', 'icicle+red', 'c5a.16xlarge', 137450),
    # 4p
    ('4p', 'changelog', 'c5a.xlarge', 105716),
    ('4p', 'changelog', 'c5a.2xlarge', 139364),
    ('4p', 'changelog', 'c5a.4xlarge', 142195),
    ('4p', 'changelog', 'c5a.8xlarge', 142262),
    ('4p', 'changelog', 'c5a.16xlarge', 139057),
    ('4p', 'icicle', 'c5a.xlarge', 91723),
    ('4p', 'icicle', 'c5a.2xlarge', 103443),
    ('4p', 'icicle', 'c5a.4xlarge', 107133),
    ('4p', 'icicle', 'c5a.8xlarge', 108428),
    ('4p', 'icicle', 'c5a.16xlarge', 109298),
    ('4p', 'icicle+red', 'c5a.xlarge', 107518),
    ('4p', 'icicle+red', 'c5a.2xlarge', 132331),
    ('4p', 'icicle+red', 'c5a.4xlarge', 140892),
    ('4p', 'icicle+red', 'c5a.8xlarge', 142490),
    ('4p', 'icicle+red', 'c5a.16xlarge', 139189),
    # 8p
    ('8p', 'changelog', 'c5a.xlarge', 103751),
    ('8p', 'changelog', 'c5a.2xlarge', 137008),
    ('8p', 'changelog', 'c5a.4xlarge', 139266),
    ('8p', 'changelog', 'c5a.8xlarge', 142754),
    ('8p', 'changelog', 'c5a.16xlarge', 139087),
    ('8p', 'icicle', 'c5a.xlarge', 89137),
    ('8p', 'icicle', 'c5a.2xlarge', 102113),
    ('8p', 'icicle', 'c5a.4xlarge', 105873),
    ('8p', 'icicle', 'c5a.8xlarge', 109354),
    ('8p', 'icicle', 'c5a.16xlarge', 108134),
    ('8p', 'icicle+red', 'c5a.xlarge', 104994),
    ('8p', 'icicle+red', 'c5a.2xlarge', 131665),
    ('8p', 'icicle+red', 'c5a.4xlarge', 138401),
    ('8p', 'icicle+red', 'c5a.8xlarge', 141682),
    ('8p', 'icicle+red', 'c5a.16xlarge', 140346),
    # 16p
    ('16p', 'changelog', 'c5a.xlarge', 100490),
    ('16p', 'changelog', 'c5a.2xlarge', 134549),
    ('16p', 'changelog', 'c5a.4xlarge', 139664),
    ('16p', 'changelog', 'c5a.8xlarge', 142536),
    ('16p', 'changelog', 'c5a.16xlarge', 137974),
    ('16p', 'icicle', 'c5a.xlarge', 87297),
    ('16p', 'icicle', 'c5a.2xlarge', 100186),
    ('16p', 'icicle', 'c5a.4xlarge', 104813),
    ('16p', 'icicle', 'c5a.8xlarge', 108991),
    ('16p', 'icicle', 'c5a.16xlarge', 107899),
    ('16p', 'icicle+red', 'c5a.xlarge', 105230),
    ('16p', 'icicle+red', 'c5a.2xlarge', 128015),
    ('16p', 'icicle+red', 'c5a.4xlarge', 136732),
    ('16p', 'icicle+red', 'c5a.8xlarge', 141853),
    ('16p', 'icicle+red', 'c5a.16xlarge', 138660),
]

df = pd.DataFrame(
    data,
    columns=['topic_partitions', 'mode', 'client_type', 'changelogs_per_s'],
)

df

In [ ]:
# Assumes:
# df columns = ["topic_partitions", "mode", "client_type", "changelogs_per_s"]
# COLUMNWIDTH_IN and HEIGHT_RATIO are already defined

# ---- Orders ----
x_labels = ['1p', '2p', '4p', '8p', '16p']
x_values = [1, 2, 4, 8, 16]

client_order = [
    'c5a.xlarge',
    'c5a.2xlarge',
    'c5a.4xlarge',
    'c5a.8xlarge',
    'c5a.16xlarge',
]

client_short_labels = ['xlarge', '2xlarge', '4xlarge', '8xlarge', '16xlarge']


# -------------------------------------------------------------------
# Plot 1: x = partitions, lines = client sizes
# -------------------------------------------------------------------
def plot_mode(mode: str, title: str, outfile: str):
    fig = plt.figure(figsize=(COLUMNWIDTH_IN, COLUMNWIDTH_IN * HEIGHT_RATIO))
    ax = plt.gca()

    for client in client_order:
        subset = df[
            (df['mode'] == mode) & (df['client_type'] == client)
        ].copy()

        subset['topic_partitions'] = pd.Categorical(
            subset['topic_partitions'],
            categories=x_labels,
            ordered=True,
        )
        subset = subset.sort_values('topic_partitions')

        ax.plot(
            x_values,
            subset['changelogs_per_s'].values,
            marker='o',
            label=client,
        )

    # ---- Axes ----
    ax.set_title(title)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Kafka Topic Partitions')
    ax.set_ylabel('Throughput\n(changelogs/s)')

    ax.set_xticks(x_values)
    ax.get_xaxis().set_major_formatter(ScalarFormatter())
    ax.set_xticklabels([str(x) for x in x_values])

    yticks = list(range(0, 150001, 25000))
    ax.set_yticks(yticks)
    ax.set_yticklabels([f'{y // 1000}K' for y in yticks])
    ax.set_ylim(0, 150000)

    # ---- Grid ----
    ax.yaxis.set_minor_locator(MultipleLocator(12500))
    ax.grid(axis='y', which='major', linestyle='--', linewidth=0.4, alpha=0.5)
    ax.grid(axis='y', which='minor', linestyle=':', linewidth=0.3, alpha=0.4)
    ax.set_axisbelow(True)

    # ---- Legend ----
    # ax.legend(frameon=False, loc="lower right")
    ax.legend(
        frameon=False,
        loc='lower right',
        ncols=2,
        bbox_to_anchor=(1.0, -0.05),
    )

    plt.tight_layout()
    plt.savefig(
        outfile,
        format='pdf',
        bbox_inches='tight',
        dpi=600,
    )


# -------------------------------------------------------------------
# Calls: partition-scaling plots
# -------------------------------------------------------------------
# plot_mode(
#     "changelog",
#     "GPFS Changelog Throughput vs. Partitions",
#     "gpfs_scaling_partition_changelog.pdf",
# )

# plot_mode(
#     "icicle",
#     "Icicle Throughput",
#     "figures/gpfs_scaling_partition_icicle.pdf",
# )

# plot_mode(
#     "icicle+red",
#     "Icicle (with Reduction) Throughput",
#     "figures/gpfs_scaling_partition_icicle_red.pdf",
# )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

# Assumes:
# df columns = ["topic_partitions", "mode", "client_type", "changelogs_per_s"]
# COLUMNWIDTH_IN and HEIGHT_RATIO are already defined

# ---- Orders ----
x_labels = ['1p', '2p', '4p', '8p', '16p']
x_values = [1, 2, 4, 8, 16]

client_order = [
    'c5a.xlarge',
    'c5a.2xlarge',
    'c5a.4xlarge',
    'c5a.8xlarge',
    'c5a.16xlarge',
]


def _prep_subset(df, mode, client):
    subset = df[(df['mode'] == mode) & (df['client_type'] == client)].copy()

    subset['topic_partitions'] = pd.Categorical(
        subset['topic_partitions'],
        categories=x_labels,
        ordered=True,
    )
    subset = subset.sort_values('topic_partitions')
    return subset


def plot_icicle_vs_red(title: str, outfile: str):
    fig = plt.figure(figsize=(COLUMNWIDTH_IN, COLUMNWIDTH_IN * HEIGHT_RATIO))
    ax = plt.gca()

    # plot each client once, with two styles (solid vs dashed) but same color
    for client in client_order:
        s1 = _prep_subset(df, 'icicle', client)
        (line_solid,) = ax.plot(
            x_values,
            s1['changelogs_per_s'].values,
            marker='o',
            linestyle='-',
            label=client,
        )
        color = line_solid.get_color()

        s2 = _prep_subset(df, 'icicle+red', client)
        ax.plot(
            x_values,
            s2['changelogs_per_s'].values,
            marker='o',
            linestyle='--',
            color=color,
            label=f'{client} (red)',  # not used in legend below, but fine to keep
        )

    # ---- Axes ----
    # ax.set_title(title)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Kafka Topic Partitions')
    ax.set_ylabel('Throughput\n(changelogs/s)')

    ax.set_xticks(x_values)
    ax.get_xaxis().set_major_formatter(ScalarFormatter())
    ax.set_xticklabels([str(x) for x in x_values])

    yticks = list(range(0, 150001, 25000))
    ax.set_yticks(yticks)
    ax.set_yticklabels([f'{y // 1000}K' for y in yticks])
    ax.set_ylim(0, 150000)

    # ---- Grid ----
    ax.yaxis.set_minor_locator(MultipleLocator(12500))
    ax.grid(axis='y', which='major', linestyle='--', linewidth=0.4, alpha=0.5)
    ax.grid(axis='y', which='minor', linestyle=':', linewidth=0.3, alpha=0.4)
    ax.set_axisbelow(True)

    # ---- Legend: (1) client colors, (2) linestyle meaning ----
    handles, labels = ax.get_legend_handles_labels()

    # keep only the first occurrence per client (the solid ones)
    client_handles = []
    client_labels = []
    seen = set()
    for h, lab in zip(handles, labels):
        if lab in client_order and lab not in seen:
            client_handles.append(h)
            client_labels.append(lab)
            seen.add(lab)

    style_handles = [
        Line2D([0], [0], linestyle='-', marker='o', label='icicle'),
        Line2D([0], [0], linestyle='--', marker='o', label='icicle+red'),
    ]

    # two stacked legends
    leg1 = ax.legend(
        client_handles,
        client_labels,
        frameon=False,
        loc='lower right',
        ncols=2,
        bbox_to_anchor=(1.0, -0.05),
        columnspacing=0.8,
        handletextpad=0.5,
        labelspacing=0.5,
    )
    ax.add_artist(leg1)

    # ax.legend(
    #     handles=style_handles,
    #     frameon=False,
    #     loc="upper left",
    # )

    plt.tight_layout()
    plt.savefig(outfile, format='pdf', bbox_inches='tight')


# Call
plot_icicle_vs_red(
    'Icicle (solid) vs. Icicle+Red. (dashed) ',
    'figures/gpfs_scaling_partition_icicle_combined.pdf',
)

In [ ]:
RAW = """
FS-small,ddsketch,p10,6,0.253,0.9,0.0051,0.011
FS-small,ddsketch,p25,6,0.262,0.75,0.0053,0.01
FS-small,ddsketch,p50,6,0.2452,0.5028,0.0048,0.0156
FS-small,ddsketch,p75,6,0.2328,0.75,0.005,0.0095
FS-small,ddsketch,p90,6,0.2058,0.9,0.0056,0.0095
FS-small,ddsketch,p99,6,0.1023,0.99,0.0057,0.0099

FS-small,kllsketch,p10,6,0.0965,0.9,0.008,0.3429
FS-small,kllsketch,p25,6,0.0886,0.75,0.003,0.2
FS-small,kllsketch,p50,6,0.1089,0.5045,0.0027,0.0568
FS-small,kllsketch,p75,6,0.1097,0.75,0.007,0.3451
FS-small,kllsketch,p90,6,0.1018,0.9009,0.0042,0.1007
FS-small,kllsketch,p99,6,0.0686,0.9892,0.0666,2.5168

FS-small,reqsketch,p10,6,0.0986,0.9,0.0317,2.0202
FS-small,reqsketch,p25,6,0.0898,0.75,0.0098,0.3131
FS-small,reqsketch,p50,6,0.1095,0.5045,0.0041,0.0882
FS-small,reqsketch,p75,6,0.1094,0.75,0.0067,0.3451
FS-small,reqsketch,p90,6,0.1013,0.9009,0.0017,0.0394
FS-small,reqsketch,p99,6,0.0705,0.9892,0.0085,0.2805

FS-small,tdigest,p10,6,0.0977,0.9,0.0194,0.8938
FS-small,tdigest,p25,6,0.0898,0.75,0.014,0.2159
FS-small,tdigest,p50,6,0.1124,0.5045,0.0132,0.2799
FS-small,tdigest,p75,6,0.1087,0.75,0.0174,0.3451
FS-small,tdigest,p90,6,0.103,0.9009,0.0096,0.2186
FS-small,tdigest,p99,6,0.0708,0.9892,0.0216,0.5354
"""
FS_MEDIUM = """
FS-medium,ddsketch,p10,6,0.21,0.9,0.008,0.5
FS-medium,ddsketch,p25,6,0.2582,0.75,0.0073,0.1667
FS-medium,ddsketch,p50,6,0.3061,0.5046,0.0066,0.0909
FS-medium,ddsketch,p75,6,0.311,0.7523,0.007,0.0909
FS-medium,ddsketch,p90,6,0.2622,0.9004,0.007,0.0588
FS-medium,ddsketch,p99,6,0.1493,0.99,0.0071,0.0189

FS-medium,kllsketch,p10,6,0.0482,0.9,0.0103,16.3583
FS-medium,kllsketch,p25,6,0.0662,0.75,0.0029,0.4996
FS-medium,kllsketch,p50,6,0.0683,0.5004,0.0027,0.3322
FS-medium,kllsketch,p75,6,0.0588,0.75,0.0052,0.973
FS-medium,kllsketch,p90,6,0.0436,0.9,0.0093,0.7509
FS-medium,kllsketch,p99,6,0.0281,0.99,0.1172,39.4836

FS-medium,reqsketch,p10,6,0.0498,0.9,0.0372,50.2992
FS-medium,reqsketch,p25,6,0.0677,0.75,0.008,2.8519
FS-medium,reqsketch,p50,6,0.0693,0.5004,0.0044,0.745
FS-medium,reqsketch,p75,6,0.059,0.75,0.0046,0.6718
FS-medium,reqsketch,p90,6,0.0431,0.9,0.004,0.7509
FS-medium,reqsketch,p99,6,0.0276,0.99,0.0319,14.0615

FS-medium,tdigest,p10,6,0.0485,0.9,0.0192,4.5
FS-medium,tdigest,p25,6,0.0693,0.75,0.0116,1.1111
FS-medium,tdigest,p50,6,0.0733,0.5004,0.0178,8.1137
FS-medium,tdigest,p75,6,0.0626,0.75,0.3747,556.0781
FS-medium,tdigest,p90,6,0.0444,0.9,0.0183,3.6322
FS-medium,tdigest,p99,6,0.0277,0.99,0.0414,14.0615
"""
FS_LARGE = """
FS-large,ddsketch,p10,6,0.2122,0.9,0.0058,0.5
FS-large,ddsketch,p25,6,0.2417,0.75,0.0051,0.0556
FS-large,ddsketch,p50,6,0.2628,0.5048,0.0052,0.1667
FS-large,ddsketch,p75,6,0.2559,0.7524,0.0053,0.1667
FS-large,ddsketch,p90,6,0.2314,0.9009,0.0054,0.1667
FS-large,ddsketch,p99,6,0.1837,0.99,0.0054,0.01

FS-large,kllsketch,p10,4,0.0374,0.9,0.0076,3.1237
FS-large,kllsketch,p25,4,0.0398,0.75,0.0056,4
FS-large,kllsketch,p50,4,0.0406,0.5023,0.1629,507.0842
FS-large,kllsketch,p75,4,0.0366,0.7511,0.0057,2.9078
FS-large,kllsketch,p90,4,0.0299,0.9005,0.0156,37.3204
FS-large,kllsketch,p99,4,0.0189,0.99,0.1623,794.1424

FS-large,reqsketch,p10,4,0.0396,0.9,0.0263,38.655
FS-large,reqsketch,p25,4,0.0413,0.75,0.0114,3.3054
FS-large,reqsketch,p50,4,0.0414,0.5023,2.2493,15604.4696
FS-large,reqsketch,p75,4,0.0365,0.7511,0.0058,2.6066
FS-large,reqsketch,p90,4,0.0294,0.9005,0.0084,6.1901
FS-large,reqsketch,p99,4,0.0182,0.99,0.0354,35.2585

FS-large,tdigest,p10,4,0.0379,0.9,0.0135,7.2478
FS-large,tdigest,p25,4,0.0422,0.75,0.023,14.3825
FS-large,tdigest,p50,4,0.0456,0.5023,1.9477,5131.0747
FS-large,tdigest,p75,4,0.0396,0.7511,0.0249,35.2701
FS-large,tdigest,p90,4,0.031,0.9005,0.0571,337.9602
FS-large,tdigest,p99,4,0.0182,0.99,0.0437,20.3447
"""

In [ ]:
from io import StringIO

import pandas as pd

cols = [
    'fs_scale',
    'algo',
    'quantile',
    'run_count',
    'rank_error_norm_mean',
    'rank_error_norm_max',
    'rel_value_error_mean',
    'rel_value_error_max',
]
df = pd.read_csv(
    StringIO(RAW + FS_MEDIUM + FS_LARGE),
    names=cols,
)
df

In [ ]:
summary = (
    df.groupby(['fs_scale', 'algo'])
    .agg(
        rank_mean_min=('rank_error_norm_mean', 'min'),
        rank_mean_max=('rank_error_norm_mean', 'max'),
        value_mean_min=('rel_value_error_mean', 'min'),
        value_mean_max=('rel_value_error_mean', 'max'),
    )
    .reset_index()
)

In [ ]:
order = ['FS-small', 'FS-medium', 'FS-large']

summary['fs_scale'] = pd.Categorical(
    summary['fs_scale'],
    categories=order,
    ordered=True,
)

summary = summary.sort_values(['fs_scale', 'algo']).reset_index(drop=True)
summary

In [ ]:
subset = df[
    (df['fs_scale'] == 'FS-large')
    & (df['algo'].isin(['ddsketch', 'kllsketch']))
]
subset

Pipeline runtime plot

In [ ]:
from io import StringIO

import pandas as pd

raw = """CPU	Filesystem	Pipeline	duration
256	FS-large	PRI	483.72
256	FS-large	CNT	629.48
256	FS-large	CNT	1,202.60
256	FS-large	SND	1,303.54
256	FS-large	SND	2,129.41
256	FS-large	SND	1,387.08
256	FS-medium	PRI	118.87
256	FS-medium	CNT	337.54
256	FS-medium	SND	608.40
256	FS-medium	SND	280.28
256	FS-small	PRI	74.55
256	FS-small	CNT	107.64
256	FS-small	SND	242.95
128	FS-large	PRI	926.83
128	FS-large	CNT	2,166.37
128	FS-large	CNT	1,387.87
128	FS-large	SND	2,480.82
128	FS-large	SND	3,307.47
128	FS-large	SND	2,764.58
128	FS-medium	PRI	132.63
128	FS-medium	CNT	354.01
128	FS-medium	SND	358.30
128	FS-medium	SND	904.33
128	FS-small	PRI	79.82
128	FS-small	CNT	166.88
128	FS-small	SND	263.11
"""

df = pd.read_csv(StringIO(raw), sep=r'\s+', engine='python')

# Clean numeric column
df['duration'] = (
    df['duration'].astype(str).str.replace(',', '', regex=False).astype(float)
)

df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MultipleLocator

# ----------------------------
# Global sizing (LaTeX widths)
# ----------------------------
PT_TO_INCH = 1 / 72.27
COLUMNWIDTH_PT = 252.0
COLUMNWIDTH_IN = COLUMNWIDTH_PT * PT_TO_INCH
HEIGHT_RATIO = 0.62  # tweak if you want taller/shorter

# ----------------------------
# Aggregate first
# ----------------------------
agg = df.groupby(['Filesystem', 'CPU', 'Pipeline'], as_index=False).agg(
    total_duration=('duration', 'sum'),
)

fs_order = ['FS-small', 'FS-medium', 'FS-large']
cpu_order = [128, 256]
pipe_order = ['PRI', 'CNT', 'SND']

agg = agg[
    agg['Filesystem'].isin(fs_order)
    & agg['CPU'].isin(cpu_order)
    & agg['Pipeline'].isin(pipe_order)
]

wide = (
    agg.pivot_table(
        index=['Filesystem', 'CPU'],
        columns='Pipeline',
        values='total_duration',
        aggfunc='sum',
        fill_value=0.0,
    )
    .reindex(
        pd.MultiIndex.from_product(
            [fs_order, cpu_order],
            names=['Filesystem', 'CPU'],
        ),
    )
    .reindex(columns=pipe_order)
)

# ----------------------------
# Plot
# ----------------------------
x = np.arange(len(fs_order))  # cluster centers
bar_w = 0.32
offsets = [-bar_w / 2, +bar_w / 2]

colors = {
    'PRI': '#4C72B0',  # blue
    'CNT': '#DD8452',  # orange
    'SND': '#55A868',  # green
}

fig, ax = plt.subplots(figsize=(COLUMNWIDTH_IN, COLUMNWIDTH_IN * HEIGHT_RATIO))

for i, cpu in enumerate(cpu_order):
    pri = np.array(
        [wide.loc[(fs, cpu), 'PRI'] for fs in fs_order],
        dtype=float,
    )
    cnt = np.array(
        [wide.loc[(fs, cpu), 'CNT'] for fs in fs_order],
        dtype=float,
    )
    snd = np.array(
        [wide.loc[(fs, cpu), 'SND'] for fs in fs_order],
        dtype=float,
    )

    xpos = x + offsets[i]

    ax.bar(
        xpos,
        pri,
        bar_w,
        color=colors['PRI'],
        label='PRI' if i == 0 else None,
    )
    ax.bar(
        xpos,
        cnt,
        bar_w,
        bottom=pri,
        color=colors['CNT'],
        label='CNT' if i == 0 else None,
    )
    ax.bar(
        xpos,
        snd,
        bar_w,
        bottom=pri + cnt,
        color=colors['SND'],
        label='SND' if i == 0 else None,
    )

# Cluster labels at FS centers
ax.set_xticks(x)
ax.set_xticklabels(fs_order)

# CPU labels under each bar: "128 KPU", "256 KPU"
for i, cpu in enumerate(cpu_order):
    for xi in x + offsets[i]:
        ax.text(
            xi,
            -0.08,
            f'{cpu} KPU',
            ha='center',
            va='top',
            transform=ax.get_xaxis_transform(),
            fontsize=9,
        )

ax.set_ylabel('Total Runtime (s)')
ax.set_title('Runtime Breakdown by Filesystem and CPU')

# If you want log scale, uncomment:
# ax.set_yscale("log")

# Minor ticks/grid every 1000 (linear scale)
# ax.yaxis.set_minor_locator(MultipleLocator(1000))
ax.grid(axis='y', which='major', linestyle='--', linewidth=0.4, alpha=0.5)
ax.grid(axis='y', which='minor', linestyle=':', linewidth=0.3, alpha=0.4)
ax.set_axisbelow(True)
ax.set_ylim(0, 20000)
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
plt.show()

In [ ]:
wide_reset = wide.reset_index()
print(wide_reset)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FixedFormatter
from matplotlib.ticker import FixedLocator
from matplotlib.ticker import MultipleLocator

# ----------------------------
# Global sizing (LaTeX widths)
# ----------------------------
PT_TO_INCH = 1 / 72.27
COLUMNWIDTH_PT = 252.0
COLUMNWIDTH_IN = COLUMNWIDTH_PT * PT_TO_INCH
HEIGHT_RATIO = 0.62

# ----------------------------
# Aggregate -> wide
# ----------------------------
agg = df.groupby(['Filesystem', 'CPU', 'Pipeline'], as_index=False).agg(
    total_duration=('duration', 'sum'),
)

fs_order = ['FS-small', 'FS-medium', 'FS-large']
cpu_order = [128, 256]
pipe_order = ['PRI', 'CNT', 'SND']

agg = agg[
    agg['Filesystem'].isin(fs_order)
    & agg['CPU'].isin(cpu_order)
    & agg['Pipeline'].isin(pipe_order)
]

wide = (
    agg.pivot_table(
        index=['Filesystem', 'CPU'],
        columns='Pipeline',
        values='total_duration',
        aggfunc='sum',
        fill_value=0.0,
    )
    .reindex(
        pd.MultiIndex.from_product(
            [fs_order, cpu_order],
            names=['Filesystem', 'CPU'],
        ),
    )
    .reindex(columns=pipe_order)
)

# ----------------------------
# Normalize by total(128 KPU) per FS
# ----------------------------
tot128 = wide.xs(128, level='CPU')[pipe_order].sum(axis=1)  # index=Filesystem

norm = wide.copy()
for fs in fs_order:
    denom = float(tot128.loc[fs]) if float(tot128.loc[fs]) != 0 else 1.0
    norm.loc[(fs, slice(None)), pipe_order] = (
        norm.loc[(fs, slice(None)), pipe_order] / denom
    )

# ----------------------------
# Plot: FS labels on top, KPU labels on bottom
# ----------------------------
colors = {'PRI': '#4C72B0', 'CNT': '#DD8452', 'SND': '#55A868'}

x = np.arange(len(fs_order))  # cluster centers
bar_w = 0.28
offsets = [-bar_w / 2, +bar_w / 2]

fig, ax = plt.subplots(figsize=(COLUMNWIDTH_IN, COLUMNWIDTH_IN * HEIGHT_RATIO))

bar_positions = []
bar_labels = []

for i, cpu in enumerate(cpu_order):
    pri = np.array(
        [norm.loc[(fs, cpu), 'PRI'] for fs in fs_order],
        dtype=float,
    )
    cnt = np.array(
        [norm.loc[(fs, cpu), 'CNT'] for fs in fs_order],
        dtype=float,
    )
    snd = np.array(
        [norm.loc[(fs, cpu), 'SND'] for fs in fs_order],
        dtype=float,
    )

    xpos = x + offsets[i]
    bar_positions.extend(xpos.tolist())
    bar_labels.extend([f'{cpu}'] * len(fs_order))

    ax.bar(
        xpos,
        pri,
        bar_w,
        color=colors['PRI'],
        label='Primary' if i == 0 else None,
    )
    ax.bar(
        xpos,
        cnt,
        bar_w,
        bottom=pri,
        color=colors['CNT'],
        label='Counting' if i == 0 else None,
    )
    ax.bar(
        xpos,
        snd,
        bar_w,
        bottom=pri + cnt,
        color=colors['SND'],
        label='Secondary' if i == 0 else None,
    )

# ---- Bottom labels: only KPU labels (minor ticks) ----
# Keep major ticks at cluster centers but hide their labels (we'll put FS labels on top instead).
ax.set_xticks(x)
ax.set_xticklabels([''] * len(fs_order))
ax.tick_params(axis='x', which='major', length=0)

ax.xaxis.set_minor_locator(FixedLocator(bar_positions))
ax.xaxis.set_minor_formatter(FixedFormatter(bar_labels))
ax.tick_params(axis='x', which='minor', labelsize=9, pad=6)

# ---- Top labels: FS label above each cluster ----
# Place using axis-fraction y coordinate so it stays near the top.
for xi, fs in zip(x, fs_order):
    ax.text(
        xi,
        -0.2,
        fs,
        ha='center',
        va='top',
        transform=ax.get_xaxis_transform(),  # x in data, y in axes fraction
        fontsize=10,
    )

ax.set_ylabel('Normalized Runtime\n(128 KPU per FS = 1)')
# ax.set_ylabel("Normalized Runtime")
ax.set_title('Normalized Runtime Breakdown')

# Y axis grid/ticks
ax.set_ylim(0, max(1.2, float(norm[pipe_order].sum(axis=1).max()) * 1.05))
ax.yaxis.set_major_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(0.1))
ax.grid(axis='y', which='major', linestyle='--', linewidth=0.4, alpha=0.5)
ax.grid(axis='y', which='minor', linestyle=':', linewidth=0.3, alpha=0.4)
ax.set_axisbelow(True)

ax.legend(
    frameon=False,
    ncol=3,
    columnspacing=0.5,
    handletextpad=0.5,
    labelspacing=0.5,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
)

# Give a bit of bottom margin for the minor tick labels
fig.tight_layout()
# plt.show()
plt.savefig('figures/runtime_breakdown.pdf', format='pdf', bbox_inches='tight')

In [ ]:
# Cluster configurations
configs = ['1 MDT\n1 client', '2 MDTs\n2 clients', '4 MDTs\n4 clients']
x = np.arange(len(configs))

# Bar layout (thinner bars)
group_width = 0.6
bar_width = group_width / 6  # 3 modes × 2 client types

# --- Throughput data ---

# c5a.large
fs_l = [1361.7, 2788.8, 6241.3]
ic_l = [5371.8, 10421.9, 22976.6]
rd_l = [5519.1, 10512.8, 23663.1]

# c5a.xlarge
fs_x = [1396.1, 2847.6, 6285.5]
ic_x = [5549.9, 10907.0, 23580.7]
rd_x = [5664.3, 11237.1, 24188.2]

plt.figure(figsize=(COLUMNWIDTH_IN, COLUMNWIDTH_IN * HEIGHT_RATIO))


# Offsets for each bar
offsets = {
    'FS_l': -2.5 * bar_width,
    'FS_x': -1.5 * bar_width,
    'IC_l': -0.5 * bar_width,
    'IC_x': 0.5 * bar_width,
    'RD_l': 1.5 * bar_width,
    'RD_x': 2.5 * bar_width,
}

# FSMonitor (orange)
plt.bar(
    x + offsets['FS_l'],
    fs_l,
    bar_width,
    color='orange',
    label='FSMonitor (c5a.large)',
)
plt.bar(
    x + offsets['FS_x'],
    fs_x,
    bar_width,
    color='orange',
    alpha=0.6,
    label='FSMonitor (c5a.xlarge)',
)

# Icicle (blue)
plt.bar(
    x + offsets['IC_l'],
    ic_l,
    bar_width,
    color='tab:blue',
    label='Icicle (c5a.large)',
)
plt.bar(
    x + offsets['IC_x'],
    ic_x,
    bar_width,
    color='tab:blue',
    alpha=0.6,
    label='Icicle (c5a.xlarge)',
)

# Icicle + Reduction (green)
plt.bar(
    x + offsets['RD_l'],
    rd_l,
    bar_width,
    color='tab:green',
    label='Icicle+Red. (c5a.large)',
)
plt.bar(
    x + offsets['RD_x'],
    rd_x,
    bar_width,
    color='tab:green',
    alpha=0.6,
    label='Icicle+Red. (c5a.xlarge)',
)

# Axes
plt.xticks(x, configs)
plt.ylabel('Throughput\n(changelogs/s)')
# plt.title("Average FileBench throughput with c5a.large and c5a.xlarge clients")
# plt.title("FSMonitor and Icicle Throughput")


# Y ticks: 5K–25K
yticks = [0, 5000, 10000, 15000, 20000, 25000, 30000]
plt.yticks(yticks, [f'{y // 1000}K' if y != 0 else '0' for y in yticks])
plt.ylim(0, 30000)

# Gridlines
plt.grid(axis='y', linestyle='--', linewidth=0.4, alpha=0.5)
plt.gca().set_axisbelow(True)

# Legend
plt.legend(
    ncol=1,
    frameon=False,
    handlelength=1.0,  # shorter line samples
    handletextpad=0.4,  # space between line and text
    labelspacing=0.2,  # vertical space between entries
    borderpad=0,  # padding inside the legend box (still applies even if frameon=False)
)

plt.tight_layout()
plt.savefig(
    'figures/lustre_scaling.pdf',
    bbox_inches='tight',
    dpi=600,
)